# Домашняя работа: SARSA и Q-learning на FrozenLake

Этот ноутбук закрепляет материалы конспектов по методам TD-control.
Нужно реализовать алгоритмы SARSA и Q-learning, сравнить их поведение и исследовать влияние модификаций среды и типа политики.


## Учебные цели
- Реализовать алгоритмы SARSA и Q-learning для поиска оптимальной политики
- Сравнить on-policy (SARSA) и off-policy (Q-learning) подходы
- Исследовать влияние модификации награды (penalty за падение в озеро)
- Изучить разницу между ε-greedy и softmax политиками
- Сформулировать выводы о применимости каждого метода


## Теоретическая справка

### SARSA (on-policy)
Обновление Q-функции происходит с использованием действия, которое реально выполняется политикой:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$$

где $a_{t+1}$ выбирается из текущей политики (например, ε-greedy).

### Q-learning (off-policy)
Обновление Q-функции использует максимальное действие независимо от политики поведения:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$$

### Ключевые различия
- **SARSA**: учитывает риски исследования (действие $a_{t+1}$ может быть случайным из-за ε)
- **Q-learning**: оценивает оптимальную политику напрямую, игнорируя исследование
- **На практике**: SARSA более консервативен, Q-learning более агрессивен


## Как выполнять работу
- Идём сверху вниз; каждый блок с `TODO` нужно заполнить своим кодом
- Если запускаете в Colab, выполните установку зависимостей
- Для воспроизводимости фиксируйте случайные сиды
- В конце заполните секцию с вопросами и выводами


### Подготовка окружения

In [ ]:
# Если работаете в Colab, раскомментируйте строки ниже
# !pip install gymnasium numpy matplotlib tqdm -q

In [ ]:
import random
from dataclasses import dataclass
from typing import Tuple, Callable

import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [ ]:
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)

## 1. Модификация среды FrozenLake

Создадим обёртку для FrozenLake, которая добавляет отрицательную награду за падение в озеро.
Это сделает задачу более реалистичной и позволит исследовать разницу между консервативным (SARSA) и агрессивным (Q-learning) поведением.

**Задача:** реализуйте `ModifiedFrozenLakeEnv` — wrapper, который:
- Возвращает `hole_penalty` (например, -1.0) при падении в дыру
- Сохраняет стандартную награду +1 за достижение цели
- Возвращает 0 для обычных шагов


In [ ]:
class ModifiedFrozenLakeEnv(gym.Wrapper):
    """Обёртка для FrozenLake с отрицательной наградой за падение в дыру."""
    
    def __init__(self, env: gym.Env, hole_penalty: float = -1.0):
        """TODO: инициализируйте wrapper и сохраните hole_penalty.
        
        Hint:
        1. Вызовите super().__init__(env)
        2. Сохраните self.hole_penalty = hole_penalty
        3. Получите карту озера: self.desc = env.unwrapped.desc
        """
        raise NotImplementedError("TODO: инициализируйте ModifiedFrozenLakeEnv")
    
    def step(self, action: int) -> Tuple[int, float, bool, bool, dict]:
        """TODO: выполните шаг и модифицируйте награду при падении в дыру.
        
        Hint:
        1. Вызовите next_state, reward, terminated, truncated, info = self.env.step(action)
        2. Проверьте, является ли next_state дырой:
           row, col = next_state // 4, next_state % 4
           is_hole = self.desc[row, col] == b'H'
        3. Если is_hole и terminated: reward = self.hole_penalty
        4. Верните (next_state, reward, terminated, truncated, info)
        """
        raise NotImplementedError("TODO: модифицируйте награды для дыр")


def make_env(seed: int = SEED, is_slippery: bool = True, hole_penalty: float = 0.0) -> gym.Env:
    """Создаёт FrozenLake с опциональной модификацией награды."""
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)
    if hole_penalty != 0.0:
        env = ModifiedFrozenLakeEnv(env, hole_penalty=hole_penalty)
    env.reset(seed=seed)
    return env


In [ ]:
# TODO: протестируйте модифицированную среду
# Создайте среду с hole_penalty=-1.0 и выполните несколько случайных эпизодов
# Убедитесь, что при падении в дыру возвращается отрицательная награда
raise NotImplementedError("TODO: протестируйте ModifiedFrozenLakeEnv")


## 2. Политики: ε-greedy и softmax

Реализуем две стратегии исследования:
1. **ε-greedy**: с вероятностью ε выбираем случайное действие, иначе — жадное
2. **Softmax (Boltzmann)**: вероятности пропорциональны $e^{Q(s,a)/\tau}$, где τ — температура


In [ ]:
def epsilon_greedy_action(Q: np.ndarray, state: int, epsilon: float) -> int:
    """TODO: выберите действие по ε-greedy политике.
    
    Hint:
    1. С вероятностью epsilon: вернуть random.randint(0, len(Q[state])-1)
    2. Иначе: вернуть np.argmax(Q[state])
    """
    raise NotImplementedError("TODO: реализуйте epsilon_greedy_action")


def softmax_action(Q: np.ndarray, state: int, temperature: float = 1.0) -> int:
    """TODO: выберите действие по softmax политике.
    
    Hint:
    1. Вычислите логиты: logits = Q[state] / temperature
    2. Нормализуйте для численной стабильности: logits = logits - np.max(logits)
    3. Вычислите вероятности: exp_values = np.exp(logits); probs = exp_values / np.sum(exp_values)
    4. Сэмплируйте: return np.random.choice(len(probs), p=probs)
    """
    raise NotImplementedError("TODO: реализуйте softmax_action")


## 3. Реализация SARSA

SARSA — on-policy алгоритм, который обновляет Q-функцию на основе реально выполненных действий:
1. Выбрать $a_t$ из текущей политики (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Выбрать $a_{t+1}$ из той же политики
4. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$

**Важно:** действие $a_{t+1}$ выбирается до обновления Q, что делает алгоритм on-policy.


In [ ]:
@dataclass
class SARSAConfig:
    gamma: float = 0.99
    alpha: float = 0.1
    epsilon: float = 0.1
    num_episodes: int = 5000
    max_steps: int = 200


def sarsa(env: gym.Env, config: SARSAConfig, use_softmax: bool = False, temperature: float = 1.0):
    """TODO: реализуйте SARSA с отслеживанием метрик.
    
    Верните: (Q, rewards_history, success_rate_history)
    где:
    - Q: np.ndarray размера (n_states, n_actions)
    - rewards_history: список средних наград за окно эпизодов
    - success_rate_history: список процента успешных эпизодов за окно
    
    Hint (алгоритм):
    1. Инициализировать Q = np.zeros((n_states, n_actions))
    2. Для каждого эпизода:
       a. state, _ = env.reset()
       b. Выбрать action из политики (epsilon_greedy_action или softmax_action)
       c. Для каждого шага до max_steps:
          - next_state, reward, terminated, truncated, _ = env.step(action)
          - Выбрать next_action из той же политики
          - Вычислить td_target:
            * Если terminated: td_target = reward
            * Иначе: td_target = reward + gamma * Q[next_state, next_action]
          - Обновить: Q[state, action] += alpha * (td_target - Q[state, action])
          - Обновить state = next_state, action = next_action
          - Прервать если terminated or truncated
    
    Hint (метрики):
    - Храните список наград за последние 100 эпизодов
    - Каждые 100 эпизодов сохраняйте среднюю награду и success rate
    - Success = эпизод закончился с reward > 0
    """
    raise NotImplementedError("TODO: реализуйте SARSA")


## 4. Реализация Q-learning

Q-learning — off-policy алгоритм, который напрямую оценивает оптимальную Q-функцию:
1. Выбрать $a_t$ из политики поведения (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$

**Ключевое отличие:** используем $\max_a Q(s_{t+1}, a)$ вместо $Q(s_{t+1}, a_{t+1})$.


In [ ]:
@dataclass
class QLearningConfig:
    gamma: float = 0.99
    alpha: float = 0.1
    epsilon: float = 0.1
    num_episodes: int = 5000
    max_steps: int = 200


def q_learning(env: gym.Env, config: QLearningConfig, use_softmax: bool = False, temperature: float = 1.0):
    """TODO: реализуйте Q-learning с отслеживанием метрик.
    
    Верните: (Q, rewards_history, success_rate_history)
    
    Hint (отличие от SARSA):
    - Не нужно выбирать next_action заранее
    - При вычислении td_target используйте:
      * Если terminated: td_target = reward
      * Иначе: td_target = reward + gamma * np.max(Q[next_state])
    - Обновление: Q[state, action] += alpha * (td_target - Q[state, action])
    
    Hint (структура):
    - Копируйте структуру из sarsa(), меняя только расчёт td_target
    - Метрики собирайте аналогично
    """
    raise NotImplementedError("TODO: реализуйте Q-learning")


## 5. Эксперимент A: SARSA vs Q-learning на стандартной FrozenLake

Сравним оба алгоритма на стандартной среде (без модификации наград).

**План:**
1. Обучить SARSA и Q-learning с одинаковыми гиперпараметрами
2. Построить графики обучения (средняя награда и success rate)
3. Сравнить финальные политики


In [ ]:
# TODO: запустите эксперимент A
#
# Hint:
# 1. Создайте среду: env = make_env(hole_penalty=0.0)
# 2. Обучите SARSA:
#    config_sarsa = SARSAConfig(num_episodes=5000)
#    Q_sarsa, rewards_sarsa, success_sarsa = sarsa(env, config_sarsa)
# 3. Обучите Q-learning:
#    config_ql = QLearningConfig(num_episodes=5000)
#    Q_ql, rewards_ql, success_ql = q_learning(env, config_ql)
# 4. Выведите финальные метрики и сравните с эталонными
raise NotImplementedError("TODO: запустите эксперимент A")


In [ ]:
# TODO: визуализируйте результаты эксперимента A
#
# Hint:
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 
# # График 1: средняя награда
# axes[0].plot(rewards_sarsa, label="SARSA", alpha=0.7)
# axes[0].plot(rewards_ql, label="Q-learning", alpha=0.7)
# axes[0].set_xlabel("Эпизоды (×100)"), axes[0].set_ylabel("Средняя награда")
# axes[0].legend(), axes[0].grid()
# 
# # График 2: success rate
# axes[1].plot(success_sarsa, label="SARSA", alpha=0.7)
# axes[1].plot(success_ql, label="Q-learning", alpha=0.7)
# axes[1].set_xlabel("Эпизоды (×100)"), axes[1].set_ylabel("Success Rate")
# axes[1].legend(), axes[1].grid()
# 
# plt.tight_layout(), plt.show()
raise NotImplementedError("TODO: визуализируйте эксперимент A")


### Выводы по эксперименту A
- TODO: Какой алгоритм достиг лучшей финальной производительности?
- TODO: Наблюдаете ли вы разницу в скорости сходимости?
- TODO: Как различаются выученные политики? (проверьте `np.argmax(Q_sarsa, axis=1)` vs `np.argmax(Q_ql, axis=1)`)

## 6. Эксперимент B: Влияние отрицательной награды за дыры

Теперь используем модифицированную среду с `hole_penalty=-1.0` и сравним поведение алгоритмов.

**Гипотеза:**
- SARSA (on-policy) будет более консервативным и избегать рискованных путей
- Q-learning (off-policy) может быть более агрессивным и выбирать опасные маршруты

In [ ]:
# TODO: запустите эксперимент B с модифицированной средой
#
# Hint:
# 1. Создайте среду: env_modified = make_env(hole_penalty=-1.0)
# 2. Обучите оба алгоритма с теми же конфигурациями
# 3. Сравните результаты с экспериментом A
# 4. Обратите внимание на среднюю награду (она будет ниже из-за penalty)
raise NotImplementedError("TODO: запустите эксперимент B")


In [ ]:
# TODO: визуализируйте сравнение экспериментов A и B
#
# Hint: создайте 2×2 сетку графиков:
# - Строка 1: стандартная среда (SARSA vs Q-learning)
# - Строка 2: модифицированная среда (SARSA vs Q-learning)
# - Колонка 1: средняя награда
# - Колонка 2: success rate
raise NotImplementedError("TODO: визуализируйте эксперимент B")


In [ ]:
# TODO: визуализируйте различия в политиках
#
# Hint: выведите карты оптимальных действий для всех 4 случаев:
# def visualize_policy(Q, title):
#     policy = np.argmax(Q, axis=1).reshape(4, 4)
#     actions = ['←', '↓', '→', '↑']
#     fig, ax = plt.subplots(figsize=(5, 5))
#     for i in range(4):
#         for j in range(4):
#             state = i * 4 + j
#             ax.text(j, i, actions[policy[i,j]], ha='center', va='center', fontsize=20)
#     ax.set_xlim(-0.5, 3.5), ax.set_ylim(-0.5, 3.5)
#     ax.set_xticks(range(4)), ax.set_yticks(range(4))
#     ax.grid(), ax.set_title(title), plt.gca().invert_yaxis()
#     plt.show()
raise NotImplementedError("TODO: визуализируйте политики")


### Выводы по эксперименту B
- TODO: Как изменилось поведение SARSA при добавлении penalty?
- TODO: Как изменилось поведение Q-learning?
- TODO: Какой алгоритм показал более безопасное поведение?
- TODO: Объясните разницу в средних наградах между алгоритмами

## 7. Эксперимент C: Softmax vs ε-greedy политика

Исследуем влияние типа политики исследования на производительность SARSA.

**Сравним:**
- ε-greedy с ε=0.1 (резкое переключение между исследованием и эксплуатацией)
- Softmax с температурой τ=0.5, 1.0, 2.0 (плавное распределение вероятностей)

In [ ]:
# TODO: запустите эксперимент C
#
# Hint:
# 1. Обучите SARSA с ε-greedy (уже сделано в эксперименте A)
# 2. Обучите SARSA с softmax и разными температурами:
#    for temp in [0.5, 1.0, 2.0]:
#        Q, rewards, success = sarsa(env, config, use_softmax=True, temperature=temp)
# 3. Соберите все метрики в словарь для сравнения
raise NotImplementedError("TODO: запустите эксперимент C")


In [ ]:
# TODO: визуализируйте сравнение политик
#
# Hint: постройте графики success rate для всех 4 вариантов на одних осях
# plt.plot(success_epsilon, label="ε-greedy (ε=0.1)")
# plt.plot(success_softmax_05, label="Softmax (τ=0.5)")
# plt.plot(success_softmax_10, label="Softmax (τ=1.0)")
# plt.plot(success_softmax_20, label="Softmax (τ=2.0)")
raise NotImplementedError("TODO: визуализируйте эксперимент C")


### Выводы по эксперименту C
- TODO: Какая температура softmax показала лучший результат?
- TODO: Как температура влияет на баланс исследования/эксплуатации?
- TODO: В каких случаях softmax предпочтительнее ε-greedy?

**Ожидаемые наблюдения:**
- Низкая температура (τ=0.5): более жадное поведение, может застрять в локальном оптимуме
- Средняя температура (τ=1.0): хороший баланс, близко к ε-greedy
- Высокая температура (τ=2.0): слишком много исследования, медленная сходимость
- Softmax более плавно переходит от исследования к эксплуатации


## 8. Дополнительный анализ (опционально)

**Задачи для углублённого изучения:**

1. **Анализ траекторий**: запишите несколько эпизодов обученных политик и визуализируйте маршруты
2. **Матрица посещений**: постройте heatmap частоты посещений состояний для разных алгоритмов
3. **Чувствительность к α**: исследуйте влияние learning rate на сходимость
4. **Double Q-learning**: реализуйте и сравните с обычным Q-learning
5. **Анализ Q-значений**: визуализируйте разницу в Q(s,a) между SARSA и Q-learning


In [ ]:
# TODO (опционально): реализуйте дополнительный анализ
pass


## 9. Вопросы для самопроверки

1. **TODO:** Почему SARSA называется on-policy, а Q-learning — off-policy? Объясните на примере.

2. **TODO:** В каких ситуациях SARSA предпочтительнее Q-learning? Приведите примеры.

3. **TODO:** Как отрицательная награда за падение в дыру влияет на поведение алгоритмов?

4. **TODO:** Почему softmax с низкой температурой может привести к худшим результатам?

5. **TODO:** Что произойдёт, если установить α=1.0? Будет ли алгоритм сходиться?